In [3]:
# ====================================================
# 🔧 STEP 1: Mount Google Drive
# ====================================================
from google.colab import drive
drive.mount('/content/drive')

# Create a working directory inside Drive
import os
WORK_DIR = '/content/drive/MyDrive/numeric_finetune_data'
os.makedirs(WORK_DIR, exist_ok=True)

Mounted at /content/drive


In [5]:
!pip install -q transformers datasets peft accelerate wandb

In [6]:
# =========================================================
# ModernBERT + LoRA + Triplet Contrastive Training
# =========================================================

import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
)

from peft import LoraConfig, get_peft_model
from accelerate import Accelerator
import wandb
from tqdm import tqdm

In [7]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


False

In [17]:
!ls drive/MyDrive/numeric_finetune_data

Numeracy_600K_comment.json  numer_sense_pairs.jsonl   trained_model
NumerSense		    numersense_random.jsonl   triplet_records.json
numersense_all_num.jsonl    rewritten_triplets.jsonl


In [18]:
! head -2 drive/MyDrive/numeric_finetune_data/NumerSense/train_extracted.jsonl


{"anchor": "$27.9M City of Middletown, Connecticut Citigroup Global Markets Inc", "positive": "$29.61M City of Middletown, Connecticut Citigroup Global Markets Inc", "negative": "$23.6M City of Middletown, Connecticut Citigroup Global Markets Inc", "number": "27.9", "positive_rewritten": "Citigroup Global Markets Inc in Middletown, Connecticut with a value of $29.61M", "negative_rewritten": "Citigroup Global Markets Inc in Middletown, Connecticut with a value of $23.6M", "positive_number": "29.61", "negative_number": 23.6}
{"anchor": "Ex-N.Y. Senate leader Bruno asks state for $2.4 million in legal fees", "positive": "Ex-N.Y. Senate leader Bruno asks state for $2.07 million in legal fees", "negative": "Ex-N.Y. Senate leader Bruno asks state for $3.36 million in legal fees", "number": "2.4", "positive_rewritten": "Former New York Senate leader Bruno requests $2.07 million from the state for legal expenses", "negative_rewritten": "Former New York Senate leader Bruno seeks $3.36 million f

In [19]:
!wc -l val_sample.jsonl

wc: val_sample.jsonl: No such file or directory


In [20]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "answerdotai/ModernBERT-base"

LORA_MODEL = "drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_corrected"

TRAIN_FILE = "drive/MyDrive/numeric_finetune_data/NumerSense/train_extracted.jsonl"     # ~79k
VAL_FILE   = "drive/MyDrive/numeric_finetune_data/NumerSense/val_extracted.jsonl"       # ~10k

MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
MARGIN = 0.2

WANDB_PROJECT = "modernbert-numeracy-lora-corrected-corrected"

In [21]:
# =========================================================
# DATASET
# =========================================================

class TripletDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor": self.tokenizer(
                item["anchor"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "positive": self.tokenizer(
                item["positive_rewritten"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "negative": self.tokenizer(
                item["negative_rewritten"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "pos_distance": abs(float(item["number"]) - float(item["positive_number"])),
            "neg_distance": abs(float(item["number"]) - float(item["negative_number"])),
        }

In [22]:
# =========================================================
# COLLATOR (DataCollatorWithPadding for triplets)
# =========================================================

def make_triplet_collator(tokenizer):
    base_collator = DataCollatorWithPadding(tokenizer)

    def collate(batch):
        return {
            "anchor": base_collator([b["anchor"] for b in batch]),
            "positive": base_collator([b["positive"] for b in batch]),
            "negative": base_collator([b["negative"] for b in batch]),
            "pos_distance": torch.tensor([b["pos_distance"] for b in batch], dtype=torch.float),
            "neg_distance": torch.tensor([b["neg_distance"] for b in batch], dtype=torch.float),
        }

    return collate

In [23]:
# =========================================================
# MEAN POOLING (IMPORTANT FOR MODERNBERT)
# =========================================================

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

In [24]:
# =========================================================
# MODEL WRAPPER
# =========================================================

class ContrastiveModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def encode(self, batch_part):
        out = self.encoder(
            input_ids=batch_part["input_ids"],
            attention_mask=batch_part["attention_mask"],
        )
        emb = mean_pooling(out, batch_part["attention_mask"])
        return F.normalize(emb, p=2, dim=1)

    def forward(self, batch):
        a = self.encode(batch["anchor"])
        p = self.encode(batch["positive"])
        n = self.encode(batch["negative"])
        return a, p, n

In [25]:
# =========================================================
# TRIPLET LOSS
# =========================================================

def triplet_loss(anchor, positive, negative, margin):
    pos_sim = F.cosine_similarity(anchor, positive)
    neg_sim = F.cosine_similarity(anchor, negative)
    return torch.mean(F.relu(neg_sim - pos_sim + margin))


In [26]:
def triplet_loss_dynamic_margin(
    anchor,
    positive,
    negative,
    pos_distance,
    neg_distance,
    base_margin=0.2,
    eps=1e-8
):
    pos_sim = F.cosine_similarity(anchor, positive)
    neg_sim = F.cosine_similarity(anchor, negative)

    # relative difficulty-aware margin
    dyn_margin = base_margin * (
        torch.log1p(neg_distance + eps) /
        torch.log1p(pos_distance + eps)
    )

    # clamp to safe range
    dyn_margin = torch.clamp(dyn_margin, min=0.0, max=base_margin)

    loss = F.relu(neg_sim - pos_sim + dyn_margin)
    return loss.mean()


In [27]:
accelerator = Accelerator()
#wandb.init(project=WANDB_PROJECT)

# Tokenizer & Base Model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)



# -----------------------------------------------------
# LoRA CONFIG (attention layers only)
# -----------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["Wqkv", "out_proj"],
    bias="none",
    task_type="FEATURE_EXTRACTION",
)

base_model = get_peft_model(base_model, lora_config)
model = ContrastiveModel(base_model)
"""
# LORA loading
# Load base architecture
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)

# Attach LoRA weights
base_model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL,
    is_trainable=True
)

model = ContrastiveModel(base_model)
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'\n# LORA loading\n# Load base architecture\nfrom peft import PeftModel\n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\nbase_model = AutoModel.from_pretrained(MODEL_NAME)\n\n# Attach LoRA weights\nbase_model = PeftModel.from_pretrained(\n    base_model,\n    LORA_MODEL,\n    is_trainable=True\n)\n\nmodel = ContrastiveModel(base_model)\n'

In [ ]:
#model.encoder.print_trainable_parameters()


In [28]:
# -----------------------------------------------------
# Data
# -----------------------------------------------------
train_dataset = TripletDataset(TRAIN_FILE, tokenizer)
val_dataset   = TripletDataset(VAL_FILE, tokenizer)

collate_fn = make_triplet_collator(tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [29]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader
)

In [ ]:
wandb.init()
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        disable=not accelerator.is_main_process,
        desc=f"Epoch {epoch+1}"
    )


    for step,batch in enumerate(progress_bar):
        a, p, n = model(batch)
        loss = triplet_loss_dynamic_margin(
                  a,
                  p,
                  n,
                  batch["pos_distance"],
                  batch["neg_distance"],
                  base_margin=0.2
              )

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        if step % 100 == 0:
          accelerator.print(f"Step {step} | Loss {loss.item():.4f}")
          wandb.log({
              "train_loss_step": loss.item(),
              "step": epoch * len(train_loader) + step
          })

    train_loss = total_loss / len(train_loader)

    # -------------------------------------------------
    # VALIDATION
    # -------------------------------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            a, p, n = model(batch)
            loss = triplet_loss_dynamic_margin(
                  a,
                  p,
                  n,
                  batch["pos_distance"],
                  batch["neg_distance"],
                  base_margin=0.2
              )
            val_loss += loss.item()

    val_loss /= len(val_loader)

    wandb.log({
         "epoch": epoch + 1,
         "train_loss": train_loss,
         "val_loss": val_loss
     })

    accelerator.print(
        f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f}"
    )

    # -----------------------------------------------------
    # SAVE MODEL AND TOKENIZER AFTER EACH EPOCH
    # -----------------------------------------------------
    accelerator.wait_for_everyone()
    unwrapped = accelerator.unwrap_model(model)
    save_path = os.path.join(WORK_DIR, f"trained_model/modernbert_lora_contrastive_corrected_dynamic_{epoch+1}")
    os.makedirs(save_path, exist_ok=True)
    unwrapped.encoder.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    accelerator.print(f"Model and tokenizer saved for epoch {epoch+1} to {save_path}")

# -----------------------------------------------------
# WANDB FINISH (after all epochs complete)
# -----------------------------------------------------
wandb.finish()

Epoch 1:   0%|          | 1/2494 [00:16<11:12:53, 16.19s/it]

Step 0 | Loss 0.1998


Epoch 1:   4%|▍         | 101/2494 [02:29<54:03,  1.36s/it]

Step 100 | Loss 0.2008


Epoch 1:   8%|▊         | 201/2494 [04:44<50:56,  1.33s/it]

...

Epoch 1:  32%|███▏      | 801/2494 [18:02<36:21,  1.29s/it]

Step 800 | Loss 0.2009


Epoch 1:  36%|███▌      | 901/2494 [20:13<33:25,  1.26s/it]

Step 900 | Loss 0.2015


Epoch 1:  38%|███▊      | 955/2494 [21:24<33:21,  1.30s/it]

In [ ]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name)


In [ ]:
!ls

sample_data


In [ ]:
accelerator.wait_for_everyone()
unwrapped = accelerator.unwrap_model(model)
unwrapped.encoder.save_pretrained("modernbert_lora_contrastive_corrected_dynamic")
tokenizer.save_pretrained("modernbert_lora_contrastive_corrected_dynamic")

wandb.finish()

In [ ]:
!ls

sample_data


In [ ]:
#!mkdir -p drive/MyDrive/numeric_finetune_data/trained_model


In [ ]:
! cp -r modernbert_lora_contrastive-corrected2 drive/MyDrive/numeric_finetune_data/trained_model/